# S1 — 약한 클래스 보강 모델 학습과 F3 앙상블

`final_model.ipynb`의 F3과 **다른 실수를 하는 모델 S1**을 무작위 초기화에서 새로 학습하고, F3과 확률을 합친다. 사전학습 가중치는 쓰지 않는다.

- 12클래스를 모두 학습하고, 약한 클래스(trash, white-glass, metal, plastic)의 손실만 2배로 한다.
- 구조: 22층(단계별 블록 2·2·3·3, 너비 48), 마지막 풀링은 평균+최대. 입력 128.
- 학습: 60 epoch, 학습률 1e-3 cosine, 넓은 자르기·모자이크·CutMix·라벨 스무딩 0.1, seed 7, 마지막 epoch 가중치.
- 결과 위치: `final_model.ipynb`와 같은 실험 폴더의 `experiments/S1` (앙상블은 `experiments/S1_ensemble`).
- 데이터 분할은 이미 만들어진 파일만 읽는다. **TEST는 쓰지 않는다.**

**실행**: Colab GPU에서 위에서부터 차례로 실행한다. 중간에 끊기면 다시 실행해 이어서 학습한다. 앙상블에는 `experiments/F3`이 필요하다.

## 1. 환경·경로·실행 스위치

In [ ]:
# [셀 목적] 라이브러리 불러오기, 프로젝트 경로·데이터 출처 경로 지정, 실행 스위치 설정
import io
import json
import random
import zlib
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter, ImageOps
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms   # 이미지 변환만 사용 (모델·사전학습 가중치는 쓰지 않음)

import shutil
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:  # Colab에는 한글 폰트가 없어 나눔고딕 설치
    import matplotlib.font_manager as fm
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], check=False, capture_output=True)
    for f in Path("/usr/share/fonts/truetype/nanum").glob("NanumGothic*.ttf"):
        fm.fontManager.addfont(str(f))
    plt.rcParams["font.family"] = "NanumGothic"
else:
    plt.rcParams["font.family"] = "Malgun Gothic"   # 한글 폰트 (Windows)
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 200)

# 실행 스위치
RUN_S1 = True            # True: S1 학습 (끝났으면 결과만 읽음, 중간에 끊겼으면 마지막으로 저장된 epoch 다음부터 이어서 학습)
RUN_S0 = False           # True: 손실 2배만 뺀 비교 모델 S0 학습 (선택, S1과 같은 시간)
RUN_ENSEMBLE = True      # True: 저장된 F3과 S1을 합쳐 val·val_ext로 평가 (TEST 사용 안 함)
S_EPOCHS = 60            # 학습 epoch 수 (바꾸면 S1_30ep처럼 새 폴더에 저장)
WEAK_CLASSES = ["trash", "white-glass", "metal", "plastic"]   # 손실을 키울 약한 클래스
WEAK_WEIGHT = 2.0        # 약한 클래스의 손실 배수
PREV_RUNS_FOLDER_ID = None   # 이전 계정 내 Drive의 garbage_classification_runs 폴더 공개 링크 ID (결과를 받아 옴)

# 데이터가 있는 Google Drive 공개 폴더 (zip 안의 경로 = 분할 CSV의 path 열)
DRIVE_FOLDER_ID = "19R4yPCNI2B204UuINfGlBx4IZ7N2j9XB"
DRIVE_NAME = "garbage_classification"
RUNS_NAME = "garbage_classification_runs"   # 다른 계정에서 실행할 때 실험 결과를 남길 내 Drive 폴더 이름
DATA_ZIPS = {"kaggle_garbage_classification.zip", "garbage_classification_v2.zip", "realwaste.zip",
             "aihub_recycling.zip", "val_hard_generated.zip", "split_and_results.zip",
             "aihub_extra.zip", "extra_public.zip", "test_dirty.zip"}   # 뒤의 두 개: 2차에서 추가한 외부 데이터


def fetch_drive_folder() -> Path:
    """Colab: Drive 폴더 위치를 돌려준다. 1순위 내 Drive 연결(폴더 주인), 2순위 공개 링크로 내려받기(gdown)"""
    src = Path("/content/drive/MyDrive") / DRIVE_NAME
    if not src.exists():
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as e:
            print("Drive 연결 생략:", e)
    if src.exists():
        return src
    return download_public_folder(DRIVE_FOLDER_ID, Path("/content/gc_drive"))


def drive_file_download(file_id: str, dst: Path, session=None) -> bool:
    """공개 파일 하나를 drive.usercontent 주소(confirm=t)로 직접 내려받는다.
    큰 파일은 Google이 '바이러스 검사 불가' 확인 페이지를 띄워 gdown이 실패하는데, confirm=t로 그 페이지를 건너뛴다.
    응답이 HTML(확인·오류 페이지)이면 저장하지 않고 False"""
    import requests
    s = session or requests.Session()
    with s.get("https://drive.usercontent.google.com/download",
               params={"id": file_id, "export": "download", "confirm": "t"}, stream=True, timeout=60) as r:
        if r.status_code != 200 or "text/html" in r.headers.get("Content-Type", ""):
            return False
        dst = Path(dst)
        dst.parent.mkdir(parents=True, exist_ok=True)
        tmp = dst.with_name(dst.name + ".part")
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
        tmp.replace(dst)
    return True


def _gdown_file(file_id: str, dst: Path) -> bool:
    """예비 경로: gdown으로 한 파일 내려받기 (실패하면 False)"""
    import gdown
    try:
        Path(dst).parent.mkdir(parents=True, exist_ok=True)
        return gdown.download(id=file_id, output=str(dst), quiet=False) is not None and Path(dst).exists()
    except Exception as e:
        print("gdown 실패:", e)
        return False


def download_public_folder(folder_id: str, out: Path, list_files=None, download=None, retries: int = 3, wait: int = 20) -> Path:
    """공개 폴더에서 필요한 파일만 내려받는다: 데이터 zip(DATA_ZIPS), 기존 실험 결과(experiments/·final_test/), 최종 가중치.
    이미 받은 파일은 건너뛰고, 실패하면 wait초씩 늘려 가며 retries번 다시 시도한다 (직접 주소 → gdown 순)"""
    out = Path(out)
    if list_files is None:
        import gdown
        list_files = lambda: gdown.download_folder(id=folder_id, output=str(out), quiet=True, skip_download=True)
    tries = [download] if download else [drive_file_download, _gdown_file]
    for f in list_files():
        name = Path(f.path).name
        parts = Path(f.path).parts
        if not (name in DATA_ZIPS or name == "final_model.pth" or (parts and parts[0] in ("experiments", "final_test"))):
            continue
        dst = Path(f.local_path)
        if dst.exists() and dst.stat().st_size > 0:
            continue
        print("내려받기:", f.path)
        for attempt in range(retries):
            if any(fn(f.id, dst) for fn in tries):
                break
            time.sleep(wait * (attempt + 1))
        else:
            raise RuntimeError(f"내려받기 실패: {f.path} — Google 다운로드 제한일 수 있음. 잠시 뒤 다시 실행하거나, "
                               f"브라우저로 공개 폴더 링크를 열어 '내 드라이브에 바로가기 추가'(이름 {DRIVE_NAME}) 후 다시 실행하세요")
    return out


def unpack_drive_data(src: Path, root: Path):
    """src의 zip을 모두 root에 풀고 최종 모델 가중치를 root/01_coding/checkpoints/에 복사 (같은 zip 목록으로 이미 풀었으면 건너뜀)"""
    done = root / ".prepared"
    zips = sorted(z for z in src.rglob("*.zip") if z.name in DATA_ZIPS)   # 알려진 데이터 zip만 (다른 대용량 zip은 무시)
    assert zips, f"데이터 zip을 찾지 못함: {src}"
    lost = sorted(DATA_ZIPS - {z.name for z in zips})
    if lost:   # 빠진 zip이 있으면 9절 실험 전에 이미지가 없다고 멈추므로 여기서 미리 알린다
        print("⚠ Drive 폴더에 없는 데이터 zip:", lost, "— 데이터 주인 Drive에 다시 올린 뒤 런타임을 초기화하고 다시 실행하세요")
    names = "\n".join(sorted(z.name for z in zips))   # .prepared에 푼 zip 이름 목록을 적어 둔다
    if done.exists() and done.read_text(encoding="utf-8") == names:
        print("데이터 준비 완료 상태:", root)
        return
    root.mkdir(parents=True, exist_ok=True)   # zip 목록이 달라졌으면(새 zip 추가 등) 다시 푼다
    for z in zips:
        print("압축 해제:", z.name)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(root)
    weights = list(src.rglob("final_model.pth"))   # 최종 모델 가중치는 있을 때만 복사 (실험 전에는 없음)
    if weights:
        (root / "01_coding" / "checkpoints").mkdir(parents=True, exist_ok=True)
        shutil.copy(weights[0], root / "01_coding" / "checkpoints" / "final_model.pth")
    else:
        print("최종 모델 가중치 없음 (실험 전이면 정상)")
    done.write_text(names, encoding="utf-8")


# 프로젝트 경로: 로컬은 저장소 루트(이 노트북 위치 = <루트>/01_coding/notebooks), Colab은 /content/garbage_classification
if IN_COLAB:
    ROOT = Path("/content") / DRIVE_NAME
    DATA_SRC = fetch_drive_folder()          # 데이터 주인 Drive 폴더 또는 공개 링크로 내려받은 폴더
    unpack_drive_data(DATA_SRC, ROOT)
else:
    ROOT = Path.cwd().resolve().parents[1]
assert (ROOT / "01_coding" / "RESULT" / "data_split").exists(), f"데이터 분할 폴더가 없음: {ROOT} (로컬은 01_coding/notebooks에서 실행)"
CODING = ROOT / "01_coding"
DATA_DIR = CODING / "RESULT" / "data_split"      # 분할·데이터 목록 CSV (고정)
MODEL_DIR = CODING / "RESULT" / "final_model"    # 최종 모델 설정·학습 기록
def results_root(in_colab: bool, drive_home: Path = Path("/content/drive/MyDrive")) -> Path:
    """실험·TEST 결과를 남길 폴더 (Colab은 VM이 초기화되면 /content가 지워지므로 Drive에 둔다)
    - 데이터 주인 계정(내 Drive에 garbage_classification 폴더가 있음): 그 폴더
    - 다른 계정(공개 링크로 데이터를 받음): 그 계정의 내 Drive/garbage_classification_runs
    - Drive가 없거나 로컬: 01_coding/RESULT"""
    if in_colab and (drive_home / DRIVE_NAME).exists():
        return drive_home / DRIVE_NAME
    if in_colab and drive_home.exists():
        return drive_home / RUNS_NAME
    return ROOT / "01_coding" / "RESULT"


def seed_results(src: Path, dst_root: Path):
    """공개 폴더에 있던 실험 결과(experiments/)와 TEST 1회 기록(final_test/)을 결과 폴더로 복사 — 이미 있는 파일은 덮어쓰지 않음.
    그래서 다른 계정에서도 1차 결과 표가 보이고, 끝난 실험은 건너뛰며, TEST 1회 보호가 유지된다"""
    src, dst_root = Path(src), Path(dst_root)
    if src.resolve() == dst_root.resolve():
        return
    n = 0
    for sub in ["experiments", "final_test"]:
        for f in (src / sub).rglob("*") if (src / sub).exists() else []:
            t = dst_root / sub / f.relative_to(src / sub)
            if f.is_file() and not t.exists():
                t.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy(f, t)
                n += 1
    print(f"기존 결과 {n}개 파일을 결과 폴더로 복사:", dst_root)


def final_test_dir(in_colab: bool) -> Path:
    """TEST 1회 결과 폴더 = 결과 폴더/final_test"""
    return results_root(in_colab) / "final_test"


if IN_COLAB:
    seed_results(DATA_SRC, results_root(True))
FINAL_DIR = final_test_dir(IN_COLAB)     # 최종 TEST 1회 결과
CKPT_PATH = CODING / "checkpoints" / "final_model.pth"   # 최종 모델 가중치 (git 제외)
PHASH_CACHE = DATA_DIR / "phash_cache.json"      # 해시 계산 캐시 (git 제외)

# 원본·외부 데이터 위치 (git 제외)
KAGGLE_DIR = ROOT / "archive" / "garbage_classification"            # Kaggle Garbage Classification 12 classes
V2_DIR = ROOT / "archive(1)" / "original"                            # Kaggle Garbage Classification V2
RW_DIR = ROOT / "realwaste" / "realwaste-main" / "RealWaste"         # RealWaste (UCI)
AIHUB_ZIP_DIR = ROOT / "232.재활용품 분류 및 선별 데이터" / "01-1.정식개방데이터"  # AI Hub 재활용품 분류 및 선별 데이터
AIHUB_OUT = ROOT / "external_data" / "aihub"      # AI Hub zip에서 꺼낸 사진 저장 위치
VAL_HARD_IMG = ROOT / "external_data" / "val_hard"  # 손상 검증셋 이미지 저장 위치

SEED = 42
CACHE_ROOT = Path("/content/cache") if IN_COLAB else ROOT / "external_data" / "cache"  # 192px로 줄인 이미지 캐시 (git 제외)
SUBSET_FOR_TEST = None   # 테스트 전용: 정수면 학습·검증 데이터를 그 장수로 줄인다
CLASSES = ["battery", "biological", "brown-glass", "cardboard", "clothes", "green-glass",
           "metal", "paper", "plastic", "shoes", "trash", "white-glass"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IMG_EXT = {".jpg", ".jpeg", ".png"}
import os
torch.set_num_threads(os.cpu_count() or 1)  # CPU 스레드 = 코어 수 (로컬 12, Colab 2)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Colab GPU가 있으면 사용


def seed_everything(seed: int = SEED):
    """random / numpy / torch 난수 시드를 모두 고정해 재현성을 확보한다."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def rel(p: Path) -> str:
    """프로젝트 루트 기준 상대경로('/' 구분) — CSV에 저장하는 경로 형식"""
    return p.relative_to(ROOT).as_posix()


def keep_available(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """이미지 파일이 실제로 있는 행만 남긴다. 빠진 사진(예: 공개 폴더에 없는 AI Hub)은 출처별 장수를 출력"""
    ok = df["path"].map(lambda p: (ROOT / p).exists())
    if not ok.all():
        print(f"[주의] {name}: 이미지 없음 {(~ok).sum()}장 제외 →", df.loc[~ok, "source"].value_counts().to_dict(),
              "(원래 결과와 점수가 달라질 수 있음)")
    return df[ok].reset_index(drop=True)


DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print("실행 환경:", "Colab" if IN_COLAB else "로컬", "/ 장치:", DEVICE)
print("프로젝트 루트:", ROOT)
print("RUN_S1 =", RUN_S1, "/ RUN_S0 =", RUN_S0, "/ RUN_ENSEMBLE =", RUN_ENSEMBLE, "/ epoch", S_EPOCHS)

## 2. 데이터 읽기

In [ ]:
# [셀 목적] 고정된 분할 파일(split.csv, *_clean.csv)을 읽어 학습·검증 데이터를 준비하고 누출이 없는지 검사 (새로 만들지 않음)
split = pd.read_csv(DATA_DIR / "split.csv")
val_df = split[split.split == "val"].reset_index(drop=True)
train_mix, val_ext, val_hard = (pd.read_csv(DATA_DIR / f"{n}_clean.csv") for n in ["train_mix", "val_ext", "val_hard"])
assert not set(train_mix.path) & (set(val_ext.path) | set(split[split.split != "train"].path)), "학습 데이터가 검증·TEST와 겹침"
assert not set(val_hard.orig_path) & set(train_mix.path), "val_hard 원본 사진이 학습 데이터에 있음"
print(f"학습 {len(train_mix):,} / val {len(val_df):,} / val_ext {len(val_ext):,} / val_hard {len(val_hard):,}")
_n = train_mix.label.value_counts()
display(pd.DataFrame({"학습 장수": _n, "손실 가중치": [WEAK_WEIGHT if c in WEAK_CLASSES else 1.0 for c in _n.index]}))

In [ ]:
# [셀 목적] 손상 검증셋 val_dirty 만들기: val_hard 원본 1,198장에 8가지 손상(약·강)을 돌아가며 입혀 저장 (있으면 읽기만)
CORRUPTIONS = ["noise", "jpeg", "motion_blur", "low_light_noise", "glare", "stain", "dark", "blur"]


def corrupt(img: Image.Image, kind: str, strong: bool, rng: np.random.Generator) -> Image.Image:
    img = img.convert("RGB")
    a = np.asarray(img, dtype=np.float32)
    if kind == "noise":
        a = a + rng.normal(0, 25 if strong else 12, a.shape)
    elif kind == "low_light_noise":
        a = a * (0.25 if strong else 0.45) + rng.normal(0, 12 if strong else 6, a.shape)
    elif kind == "dark":
        a = a * (0.3 if strong else 0.5)
    elif kind == "glare":
        h, w = a.shape[:2]
        cy, cx = rng.uniform(0.2, 0.8) * h, rng.uniform(0.2, 0.8) * w
        yy, xx = np.mgrid[:h, :w]
        r = (0.45 if strong else 0.3) * max(h, w)
        g = np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * r ** 2))[..., None]
        a = a + g * (200 if strong else 130)
    elif kind == "stain":
        out = img.copy()
        d = ImageDraw.Draw(out, "RGBA")
        h, w = a.shape[:2]
        for _ in range(4 if strong else 2):
            rx, ry = rng.uniform(0.08, 0.18) * w, rng.uniform(0.08, 0.18) * h
            cx, cy = rng.uniform(0.1, 0.9) * w, rng.uniform(0.1, 0.9) * h
            col = tuple(int(v) for v in rng.integers(40, 110, 3)) + (170 if strong else 120,)
            d.ellipse([cx - rx, cy - ry, cx + rx, cy + ry], fill=col)
        return out
    else:
        if kind == "jpeg":
            buf = io.BytesIO()
            img.save(buf, "JPEG", quality=8 if strong else 20)
            return Image.open(buf).convert("RGB")
        if kind == "blur":
            return img.filter(ImageFilter.GaussianBlur(4 if strong else 2))
        if kind == "motion_blur":
            k = 15 if strong else 9
            kern = np.zeros((k, k), np.float32)
            kern[k // 2] = 1
            ang = rng.uniform(0, 180)
            kim = Image.fromarray((kern * 255).astype(np.uint8)).rotate(ang)
            kern = np.asarray(kim, np.float32)
            kern /= kern.sum()
            from scipy.signal import fftconvolve
            a = np.stack([fftconvolve(a[..., c], kern, mode="same") for c in range(3)], -1)
        else:
            raise ValueError(kind)
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))


def build_val_dirty(src: pd.DataFrame, out_root: Path, seed: int = 7) -> pd.DataFrame:
    """src(val_hard)의 원본(orig_path)마다 손상 하나: 종류는 8가지를 돌아가며, 세기는 종류마다 약·강 번갈아.
    짧은 변 384px로 줄인 뒤 손상을 입혀 out_root/<클래스>/에 저장. 이미 있는 파일은 다시 만들지 않는다"""
    rng = np.random.default_rng(seed)
    rows, missing = [], 0
    for k, r in enumerate(src.sort_values("path").itertuples()):
        kind, strong = CORRUPTIONS[k % len(CORRUPTIONS)], bool((k // len(CORRUPTIONS)) % 2)
        dst = Path(out_root) / r.label / f"{k:04d}_{kind}_{'strong' if strong else 'mild'}.png"
        sub = np.random.default_rng(rng.integers(1 << 31))     # 사진마다 고정된 난수 (있는 파일을 건너뛰어도 다음 사진이 같음)
        if not dst.exists():
            if not (ROOT / r.orig_path).exists():   # 원본이 없으면(예: AI Hub 미공개) 건너뜀
                missing += 1
                continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            im = ImageOps.exif_transpose(Image.open(ROOT / r.orig_path)).convert("RGB")
            s = 384 / min(im.size)
            if s < 1:
                im = im.resize((round(im.width * s), round(im.height * s)), Image.BILINEAR)
            corrupt(im, kind, strong, sub).save(dst)
        rows.append(dict(path=dst.relative_to(ROOT).as_posix(), label=r.label, source=r.source, split="val_dirty",
                         orig_path=r.orig_path, corruption=kind, severity="strong" if strong else "mild"))
    if missing:
        print(f"⚠ 원본이 없어 손상 검증셋에서 뺀 사진 {missing}장 (실험 사다리는 전체 데이터가 있을 때만 실행됨)")
    return pd.DataFrame(rows)


val_dirty = build_val_dirty(val_hard, ROOT / "external_data" / "val_dirty")
print("손상 검증셋:", len(val_dirty), "장")
display(val_dirty.groupby(["corruption", "severity"]).size().unstack())

## 3. 이미지 캐시·전처리·증강

In [ ]:
# [셀 목적] 192px 이미지 캐시 함수(찌그러뜨리기·비율 유지), 정규화 통계(정리된 학습 데이터), 증강 transform, Dataset·DataLoader 정의
CACHE_SIZE = 192


def letterbox(img: Image.Image, size: int, fill: tuple) -> Image.Image:
    """비율을 유지한 채 긴 변을 size로 맞추고, 남는 부분은 fill 색으로 채워 size×size 정사각형 만들기(사진은 가운데)"""
    w, h = img.size
    r = size / max(w, h)
    img = img.resize((max(1, round(w * r)), max(1, round(h * r))), Image.BILINEAR)
    canvas = Image.new("RGB", (size, size), tuple(fill))
    canvas.paste(img, ((size - img.width) // 2, (size - img.height) // 2))
    return canvas


def object_box(img: Image.Image) -> tuple:
    """물체 상자 추정 (0~1 비율 x0, y0, x1, y1).
    1) 64px로 줄여 흐린 뒤, 테두리 띠(각 변 5px)의 색 중앙값을 배경색으로 본다
    2) 문턱값 = max(40, 테두리 픽셀 배경색 거리의 75백분위 × 2) — 아스팔트처럼 거친 바닥이면 문턱이 올라가고, 테두리에 물체가 조금 걸쳐도 흔들리지 않는다
    3) 문턱을 넘는 덩어리 중 가장 큰 것의 5% 이상인 덩어리를 씨앗으로, 문턱 × 0.6을 넘으면서 씨앗과 이어진 부분까지 물체로 본다
    4) 물체 픽셀이 0.5% 미만이거나 85% 초과면(배경이 복잡한 사진 등) 가운데 60%를 물체로 본다. 마지막에 각 변 10% 넓힌다
    가는 부분(병 목 등)이 배경과 색이 비슷하면 상자 밖으로 빠질 수 있다 → bg_replace는 상자 바깥을 완전히 바꾸지 않고 섞는다"""
    from scipy import ndimage
    a = np.asarray(img.convert("RGB").resize((64, 64)).filter(ImageFilter.GaussianBlur(2)), dtype=np.float32)
    band = np.concatenate([a[:5].reshape(-1, 3), a[-5:].reshape(-1, 3), a[:, :5].reshape(-1, 3), a[:, -5:].reshape(-1, 3)])
    bg = np.median(band, 0)
    bd = np.linalg.norm(band - bg, axis=-1)
    thr = max(40.0, float(np.percentile(bd, 75)) * 2)   # 75백분위: 테두리의 4분의 1까지 물체가 걸쳐도 문턱이 흔들리지 않음
    dist = np.linalg.norm(a - bg, axis=-1)
    strong = ndimage.binary_opening(dist > thr, iterations=1)
    lab, n = ndimage.label(strong)
    keep = np.zeros_like(strong)
    if n:
        sizes = ndimage.sum(strong, lab, range(1, n + 1))
        seed = np.isin(lab, 1 + np.nonzero(sizes >= 0.05 * sizes.max())[0])
        weak = dist > thr * 0.6
        wl, _ = ndimage.label(weak)
        keep = (np.isin(wl, np.unique(wl[seed & weak])) & weak) | seed
    if not 0.005 <= keep.mean() <= 0.85:
        x0, y0, x1, y1 = 0.2, 0.2, 0.8, 0.8
    else:
        ys, xs = np.nonzero(keep)
        x0, x1, y0, y1 = xs.min() / 64, (xs.max() + 1) / 64, ys.min() / 64, (ys.max() + 1) / 64
    return tuple(float(np.clip(v, 0, 1)) for v in (x0 - 0.1, y0 - 0.1, x1 + 0.1, y1 + 0.1))


def bg_replace(img: Image.Image, box, donor, mode: str, strength=None) -> Image.Image:
    """물체 상자 바깥만 바꾼다 (상자 안은 그대로, 경계 8px은 부드럽게 섞음).
    상자 바깥은 원래 사진을 완전히 지우지 않고 strength(기본: 0.6~0.9 무작위)만큼 새 배경으로 덮는다
    — 상자 추정이 물체 일부를 놓쳐도 그 부분이 흐리게 남도록.
    mode: swap(다른 학습 사진 donor의 같은 위치) / noise(무작위 잡음) / blur(강하게 흐림) / solid(무작위 단색)"""
    img = img.convert("RGB")
    w, h = img.size
    if isinstance(box, str):
        box = tuple(float(v) for v in box.split(","))
    if mode == "swap":
        bg = donor.convert("RGB").resize((w, h))
    elif mode == "noise":
        bg = Image.fromarray(np.random.default_rng(random.getrandbits(32)).integers(0, 256, (h, w, 3), dtype=np.uint8))
    elif mode == "blur":
        bg = img.resize((max(1, w // 4), max(1, h // 4))).filter(ImageFilter.GaussianBlur(2)).resize((w, h))   # 작게 흐려 빠르게
    elif mode == "gray":     # 배경 의존도 측정용: 상자 바깥을 회색(128)으로
        bg = Image.new("RGB", (w, h), (128, 128, 128))
    elif mode == "solid":
        bg = Image.new("RGB", (w, h), tuple(random.randint(0, 255) for _ in range(3)))
    else:
        raise ValueError(mode)
    x0, y0, x1, y1 = box[0] * w, box[1] * h, box[2] * w, box[3] * h
    xs, ys = np.arange(w, dtype=np.float32) + 0.5, np.arange(h, dtype=np.float32) + 0.5
    dx = np.maximum(np.maximum(x0 - xs, xs - x1), 0)[None, :]     # 상자까지 가로 거리 (1차원 → 2차원 broadcast)
    dy = np.maximum(np.maximum(y0 - ys, ys - y1), 0)[:, None]
    dist = np.sqrt(dx * dx + dy * dy)
    k = random.uniform(0.6, 0.9) if strength is None else strength
    alpha = 1 - k * (1 - np.clip(1 - dist / 8, 0, 1)[..., None])   # 상자 안 1, 바깥으로 8px에 걸쳐 1-k
    alpha = alpha.astype(np.float32)
    out = np.asarray(img, dtype=np.float32) * alpha + np.asarray(bg, dtype=np.float32) * (1 - alpha)
    return Image.fromarray(np.round(out).astype(np.uint8))


BG_MODES, BG_WEIGHTS = ["swap", "noise", "blur", "solid"], [0.6, 0.15, 0.15, 0.1]


def cache_resized(df: pd.DataFrame, cache_root: Path, size: int = CACHE_SIZE, name: str = "데이터",
                  mode: str = "squash") -> pd.DataFrame:
    """각 이미지를 RGB size×size PNG로 저장(이미 있으면 건너뜀). file 열(경로)을 붙여 반환.
    mode="squash": 찌그러뜨려 맞춤 → cache_root/<size>/<path>.png (1차 방식)
    mode="letterbox": 비율 유지 + 여백 채우기 → cache_root/<size>_lb/<path>.png"""
    assert mode in ("squash", "letterbox"), mode
    df = keep_available(df, name)            # 없는 파일(예: 내려받지 않은 AI Hub)은 제외하고 경고 출력
    folder = str(size) if mode == "squash" else f"{size}_lb"
    files = []
    for p in df["path"]:
        dst = Path(cache_root) / folder / (p + ".png")
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            img = Image.open(ROOT / p).convert("RGB")
            img = img.resize((size, size), Image.BILINEAR) if mode == "squash" else letterbox(img, size, FILL)
            img.save(dst)
        files.append(str(dst))
    box_csv = Path(cache_root) / folder / "boxes.csv"   # 사진별 물체 상자 (배경 교란·배경 의존도에 씀)
    boxes = dict(pd.read_csv(box_csv).values) if box_csv.exists() else {}
    new = [(p, f) for p, f in zip(df["path"], files) if p not in boxes]
    for p, f in new:
        with Image.open(f) as im:
            boxes[p] = ",".join(f"{v:.3f}" for v in object_box(im))
    if new:
        pd.DataFrame(list(boxes.items()), columns=["path", "box"]).to_csv(box_csv, index=False)
    return df.assign(file=files, box=[boxes[p] for p in df["path"]])


# 정규화 통계: 정리된 학습 데이터 2,000장(seed 42)을 64px로 줄여 RGB 평균·표준편차를 구해 json으로 저장
# (1차의 norm_stats.json은 그대로 두고, 정리된 데이터용은 norm_stats_clean.json에 따로 둔다)
stats_path = DATA_DIR / "norm_stats_clean.json"
if stats_path.exists():
    MEAN, STD = json.load(open(stats_path, encoding="utf-8")).values()
else:
    _px = np.concatenate([np.asarray(Image.open(ROOT / p).convert("RGB").resize((64, 64)), dtype=np.float32).reshape(-1, 3) / 255
                          for p in keep_available(train_mix, "학습 데이터").sample(2000, random_state=SEED).path])
    MEAN = [round(float(v), 4) for v in _px.mean(0)]
    STD = [round(float(v), 4) for v in _px.std(0)]
    json.dump({"mean": MEAN, "std": STD}, open(stats_path, "w", encoding="utf-8"))
FILL = tuple(int(round(m * 255)) for m in MEAN)   # 여백·회전으로 생기는 빈 모서리를 채울 색(평균 색)
print("정규화 평균", MEAN, "표준편차", STD, "여백 색", FILL)


class GaussianNoise:
    """사진 잡음: 픽셀마다 정규분포 잡음(표준편차 sigma 범위에서 무작위)"""
    def __init__(self, sigma=(5, 25)):
        self.sigma = sigma

    def __call__(self, img):
        a = np.asarray(img, dtype=np.float32)
        a = a + np.random.default_rng(random.getrandbits(32)).normal(0, random.uniform(*self.sigma), a.shape)
        return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))

    def __repr__(self):
        return f"GaussianNoise(sigma={self.sigma})"


class JpegCompress:
    """강한 JPEG 압축: 품질 quality 범위에서 무작위로 저장했다가 다시 읽음 (네모난 깨짐)"""
    def __init__(self, quality=(8, 40)):
        self.quality = quality

    def __call__(self, img):
        buf = io.BytesIO()
        img.convert("RGB").save(buf, "JPEG", quality=random.randint(*self.quality))
        return Image.open(buf).convert("RGB")

    def __repr__(self):
        return f"JpegCompress(quality={self.quality})"


class Glare:
    """빛번짐: 무작위 위치·반경의 밝은 원형 반점을 더함 (세기 strength 범위)"""
    def __init__(self, strength=(80, 200)):
        self.strength = strength

    def __call__(self, img):
        a = np.asarray(img, dtype=np.float32)
        h, w = a.shape[:2]
        cy, cx = random.uniform(0.2, 0.8) * h, random.uniform(0.2, 0.8) * w
        r = random.uniform(0.2, 0.45) * max(h, w)
        yy, xx = np.mgrid[:h, :w]
        g = np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * r ** 2))[..., None]
        return Image.fromarray(np.clip(a + g * random.uniform(*self.strength), 0, 255).astype(np.uint8))

    def __repr__(self):
        return f"Glare(strength={self.strength})"


class Stain:
    """얼룩: 어두운 반투명 타원 n개 (불투명도 alpha 범위)"""
    def __init__(self, n=(1, 4), alpha=(100, 180)):
        self.n, self.alpha = n, alpha

    def __call__(self, img):
        out = img.convert("RGB").copy()
        d = ImageDraw.Draw(out, "RGBA")
        w, h = out.size
        for _ in range(random.randint(*self.n)):
            rx, ry = random.uniform(0.08, 0.18) * w, random.uniform(0.08, 0.18) * h
            cx, cy = random.uniform(0.1, 0.9) * w, random.uniform(0.1, 0.9) * h
            col = tuple(random.randint(40, 110) for _ in range(3)) + (random.randint(*self.alpha),)
            d.ellipse([cx - rx, cy - ry, cx + rx, cy + ry], fill=col)
        return out

    def __repr__(self):
        return f"Stain(n={self.n}, alpha={self.alpha})"


class LowLightNoise:
    """저조도: 밝기를 scale배로 낮추고 잡음을 더함 (어두운 곳에서 찍은 사진)"""
    def __init__(self, scale=(0.25, 0.5), sigma=(5, 12)):
        self.scale, self.sigma = scale, sigma

    def __call__(self, img):
        a = np.asarray(img, dtype=np.float32) * random.uniform(*self.scale)
        a = a + np.random.default_rng(random.getrandbits(32)).normal(0, random.uniform(*self.sigma), a.shape)
        return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))

    def __repr__(self):
        return f"LowLightNoise(scale={self.scale}, sigma={self.sigma})"


def build_transforms(img_size: int, aug: list):
    """aug 키워드(위에서부터 이 순서로 적용):
    모양 — 'rrc'(무작위 자르기, 1차 기록용), 'rrc_wide'(절반 확률로 넓이 15~100% 자르기), 'hflip'(좌우반전), 'vflip'(상하반전), 'rot'(±20도 회전)
    색 — 'color'(밝기·대비·채도·색조 약간 흔들기), 'trivial'(TrivialAugmentWide, 무작위 변형 하나), 'dark_blur'(어둡게 또는 흐림), 'robust'(잡음·JPEG·빛번짐·얼룩·저조도 중 하나)
    반환 (학습 transform, 평가 transform)"""
    tf = [transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)) if "rrc" in aug else transforms.Resize((img_size, img_size))]
    if "rrc_wide" in aug:   # 절반은 넓게 자르고 절반은 원본 그대로 (원본 정확도를 잃지 않게)
        tf.insert(0, transforms.RandomApply([transforms.RandomResizedCrop(img_size, scale=(0.15, 1.0))], p=0.5))
    if "hflip" in aug:
        tf.append(transforms.RandomHorizontalFlip())
    if "vflip" in aug:
        tf.append(transforms.RandomVerticalFlip())
    if "rot" in aug:
        tf.append(transforms.RandomRotation(20, fill=FILL))
    if "color" in aug:
        tf.append(transforms.ColorJitter(0.2, 0.2, 0.2, 0.03))
    if "trivial" in aug:
        tf.append(transforms.TrivialAugmentWide(fill=list(FILL)))
    if "dark_blur" in aug:
        tf.append(transforms.RandomApply([transforms.RandomChoice([
            transforms.ColorJitter(brightness=(0.3, 1.0)),   # 어둡게: 밝기 30~100%
            transforms.GaussianBlur(5, sigma=(0.5, 2.0)),    # 흐림
        ])], p=0.5))
    if "robust" in aug:   # 손상 증강: 학습에 없던 잡음·JPEG·빛번짐·얼룩·저조도 중 하나를 절반 확률로
        tf.append(transforms.RandomApply([transforms.RandomChoice([
            GaussianNoise(), JpegCompress(), Glare(), Stain(), LowLightNoise()])], p=0.5))
    norm = [transforms.ToTensor(), transforms.Normalize(MEAN, STD)]
    return transforms.Compose(tf + norm), transforms.Compose([transforms.Resize((img_size, img_size))] + norm)


class GarbageDataset(Dataset):
    """df의 file 열(캐시 이미지 경로)과 label 로 (이미지 텐서, 클래스 번호)"""
    def __init__(self, df: pd.DataFrame, transform=None, bg_swap: bool = False, bg_gray: bool = False):
        self.files = df["file"].tolist()
        self.labels = [CLASS_TO_IDX[c] for c in df["label"]]
        self.transform = transform
        self.bg_swap = bg_swap                     # 배경 교란: 모든 사진에 같은 확률(0.5), 사진마다 다른 것은 물체 상자뿐
        self.bg_gray = bg_gray                     # 평가용: 상자 바깥을 항상 회색으로 (배경 의존도 bg_drop)
        self.boxes = df["box"].tolist() if (bg_swap or bg_gray) else None

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert("RGB")
        if self.bg_gray:
            img = bg_replace(img, self.boxes[i], None, "gray", strength=1.0)
        elif self.bg_swap and random.random() < 0.5:
            mode = random.choices(BG_MODES, BG_WEIGHTS)[0]
            donor = Image.open(self.files[random.randrange(len(self.files))]) if mode == "swap" else None
            img = bg_replace(img, self.boxes[i], donor, mode)
        return (self.transform(img) if self.transform else img), self.labels[i]


def make_loader(df, tf, batch, shuffle, bg_swap: bool = False, bg_gray: bool = False):
    workers = 2 if IN_COLAB else 0   # Colab은 CPU 2개로 이미지 읽기를 병렬화, Windows 로컬은 0
    return DataLoader(GarbageDataset(df, tf, bg_swap, bg_gray), batch_size=batch, shuffle=shuffle, num_workers=workers,
                      pin_memory=DEVICE.type == "cuda", persistent_workers=workers > 0)


print(build_transforms(128, ["hflip", "vflip", "rot", "color", "trivial", "dark_blur", "robust"])[0])

## 4. 모델 (평균+최대 풀링 추가)

In [ ]:
# [셀 목적] 잔차 블록·작은 ResNet 직접 구현(사전학습 없음), 설정(cfg)으로 모델을 만드는 함수 정의 + 파라미터 수 확인
class BasicBlock(nn.Module):
    """3×3 합성곱 2개 + 지름길(identity, 크기·채널이 바뀌면 1×1 합성곱)"""
    def __init__(self, cin: int, cout: int, stride: int):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.shortcut = nn.Sequential()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return torch.relu(out + self.shortcut(x))


class SmallResNet(nn.Module):
    """줄기 → 4단계(채널 w,2w,4w,8w × 블록 2개) → 전역 평균 풀링 → 헤드"""
    def __init__(self, width: int = 32, hidden: tuple = (), num_classes: int = 12, depths: tuple = (2, 2, 2, 2), pool: str = "avg"):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, width, 3, 2, 1, bias=False), nn.BatchNorm2d(width), nn.ReLU())
        blocks, cin = [], width
        for i, mult in enumerate([1, 2, 4, 8]):
            for j in range(depths[i]):
                blocks.append(BasicBlock(cin, width * mult, 2 if (j == 0 and i > 0) else 1))
                cin = width * mult
        self.layers = nn.Sequential(*blocks)
        self.depths = tuple(depths)
        self.pool_type = pool
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.maxpool = nn.AdaptiveMaxPool2d(1)
        head, d = [], cin * (2 if pool == "avgmax" else 1)   # avgmax: 평균과 최대를 이어 붙임
        for h in hidden:
            head += [nn.Linear(d, h), nn.ReLU()]
            d = h
        self.fc = nn.Sequential(*head, nn.Linear(d, num_classes))
        for m in self.modules():  # 합성곱은 He(kaiming) 초기화
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")

    def forward(self, x):
        f = self.layers(self.stem(x))
        p = torch.flatten(self.pool(f), 1)
        if self.pool_type == "avgmax":
            p = torch.cat([p, torch.flatten(self.maxpool(f), 1)], dim=1)
        return self.fc(p)


def build_model(cfg: dict) -> nn.Module:
    return SmallResNet(width=cfg.get("width", 32), hidden=tuple(cfg.get("hidden", ())),
                       depths=tuple(cfg.get("depths", (2, 2, 2, 2))), pool=cfg.get("pool", "avg"))


_m = SmallResNet(width=48, depths=(2, 2, 3, 3), pool="avgmax")
print(_m.fc)
print(f"파라미터 {sum(p.numel() for p in _m.parameters()):,}개 (모두 학습, 사전학습 없음)")

## 5. 학습 함수
정답 라벨이 약한 클래스면 손실을 `WEAK_WEIGHT`배 한다(`CrossEntropyLoss(weight=...)`). 평가 손실은 가중치 없음. epoch마다 `last.ckpt`를 저장해 끊기면 이어서 학습한다.

In [ ]:
# [셀 목적] 기본 설정(BASE_CFG), 실험 폴더 위치, 1 epoch 학습·평가 함수, 실험 하나를 실행·저장·이어하기 하는 run_experiment 정의
BASE_CFG = dict(width=32, hidden=(), img_size=128, aug=[], train_data="kaggle", max_per_class=1000,
                epochs=30, patience=7, batch=64, lr=1e-3, weight_decay=1e-4, scheduler="none",
                label_smoothing=0.0, seed=42)


def experiment_root(in_colab: bool) -> Path:
    """실험 폴더 = 결과 폴더/experiments (데이터 주인은 Drive의 garbage_classification, 다른 계정은 garbage_classification_runs)"""
    return results_root(in_colab) / "experiments"


def load_train_df(cfg: dict) -> pd.DataFrame:
    """train_data='kaggle': Kaggle train에서 클래스당 최대 max_per_class장(seed) / 'mix2': 정리된 혼합 학습 데이터 27,050장 / 'mix': 1차 혼합 학습 데이터"""
    if cfg["train_data"] == "mix2":
        return train_mix
    k = split[split.split == "train"]
    return pd.concat([g.sample(n=min(len(g), cfg["max_per_class"]), random_state=cfg["seed"])
                      for _, g in k.groupby("label")]).reset_index(drop=True)


def make_criteria(cfg: dict, train_df: pd.DataFrame):
    """(학습용 손실, 평가용 손실). 평가용은 항상 가중치 없음 — 실험끼리 같은 잣대로 비교하기 위해"""
    weight = None
    if cfg.get("weak_weight", 1.0) != 1.0:   # 약한 클래스(정답 라벨 기준)의 손실만 weak_weight배, 나머지는 1배
        weight = torch.tensor([cfg["weak_weight"] if c in cfg["weak_classes"] else 1.0 for c in CLASSES]).to(DEVICE)
    train_criterion = nn.CrossEntropyLoss(weight=weight, label_smoothing=cfg["label_smoothing"])
    return train_criterion, nn.CrossEntropyLoss()


def _subset(df: pd.DataFrame, seed: int) -> pd.DataFrame:
    """테스트 전용: SUBSET_FOR_TEST가 정수면 그 장수만 무작위로 남김"""
    return df.sample(n=min(SUBSET_FOR_TEST, len(df)), random_state=seed) if SUBSET_FOR_TEST else df


def mosaic_same_class(x, y):
    """같은 클래스 사진 4장을 2×2로 붙인 뒤(256) 무작위 위치를 원래 크기(128)로 잘라 냄. 라벨은 그대로 (배치에 같은 클래스가 없으면 자기 자신)"""
    _, _, h, w = x.shape
    same = (y[:, None] == y[None, :]).float()
    idx = torch.multinomial(same, 3, replacement=True)
    top = torch.cat([x, x[idx[:, 0]]], dim=3)
    bottom = torch.cat([x[idx[:, 1]], x[idx[:, 2]]], dim=3)
    canvas = torch.cat([top, bottom], dim=2)
    t, l = random.randint(0, h), random.randint(0, w)
    return canvas[:, :, t:t + h, l:l + w]


def cutmix(x, y):
    """다른 사진의 사각형 조각을 붙임. 반환: 섞은 사진, 붙인 사진의 라벨, 원래 사진의 넓이 비율(0.5 이상)"""
    n, _, h, w = x.shape
    perm = torch.randperm(n, device=x.device)
    side = random.uniform(0.0, 0.5) ** 0.5
    ch, cw = int(h * side), int(w * side)
    t, l = random.randint(0, h - ch), random.randint(0, w - cw)
    out = x.clone()
    out[:, :, t:t + ch, l:l + cw] = x[perm, :, t:t + ch, l:l + cw]
    return out, y[perm], 1 - ch * cw / (h * w)


def train_one_epoch(model, loader, criterion, optimizer, scaler, batch_mix: bool = False):
    model.train()
    n, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=scaler is not None):
            y2, lam = None, 1.0
            if batch_mix:                       # 배치마다: 25% 모자이크, 25% CutMix, 50% 그대로
                r = random.random()
                if r < 0.25:
                    x = mosaic_same_class(x, y)
                elif r < 0.5:
                    x, y2, lam = cutmix(x, y)
            out = model(x)
            loss = criterion(out, y) if y2 is None else lam * criterion(out, y) + (1 - lam) * criterion(out, y2)
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        loss_sum += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        n += x.size(0)
    return loss_sum / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion):
    """반환: 평균 손실, 정확도, 예측 번호 배열, 정답 번호 배열"""
    model.eval()
    n, correct, loss_sum, preds, labels = 0, 0, 0.0, [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x).float()
        loss_sum += criterion(out, y).item() * x.size(0)
        p = out.argmax(1)
        correct += (p == y).sum().item()
        n += x.size(0)
        preds.append(p.cpu())
        labels.append(y.cpu())
    return loss_sum / n, correct / n, torch.cat(preds).numpy(), torch.cat(labels).numpy()


def extra_metrics(model, cfg: dict, base: dict) -> dict:
    """3차 사다리용 지표: val_dirty(손상 검증셋 정확도), val_avg4, bg_drop(val·val_ext에서 물체 상자 바깥을 회색으로 지웠을 때 떨어지는 정확도)"""
    _, eval_tf = build_transforms(cfg["img_size"], [])
    mode = cfg.get("resize", "squash")

    def acc(df, name, gray=False):
        d = cache_resized(_subset(df, cfg["seed"]), CACHE_ROOT, name=name, mode=mode).reset_index(drop=True)
        return evaluate(model, make_loader(d, eval_tf, cfg["batch"] * 2, False, bg_gray=gray), nn.CrossEntropyLoss())[1]

    vd = acc(val_dirty, "val_dirty")
    gray = (acc(val_df, "val", True) + acc(val_ext, "val_ext", True)) / 2
    return dict(val_dirty=vd, val_avg4=(base["val"] + base["val_ext"] + base["val_hard"] + vd) / 4,
                bg_drop=(base["val"] + base["val_ext"]) / 2 - gray)


def run_experiment(exp_id: str, cfg: dict, exp_root: Path) -> dict:
    """실험 하나: 학습(epoch마다 중간 저장) → 마지막 epoch 가중치로 val·val_ext·val_hard 평가 → metrics.json. 이미 끝난 실험은 결과만 읽음"""
    out = Path(exp_root) / exp_id
    if (out / "metrics.json").exists():
        saved = json.load(open(out / "cfg.json", encoding="utf-8"))
        if saved != json.loads(json.dumps(cfg)):   # 같은 ID라도 설정이 바뀌었으면 결과를 섞지 않는다
            raise ValueError(f"{exp_id}: 저장된 결과와 설정이 다름 → {out} 폴더를 지우거나 새 ID를 쓰세요\n저장: {saved}\n현재: {cfg}")
        return json.load(open(out / "metrics.json", encoding="utf-8"))
    out.mkdir(parents=True, exist_ok=True)
    ckpt_path = out / "last.ckpt"
    if ckpt_path.exists():
        saved = json.load(open(out / "cfg.json", encoding="utf-8"))
        assert saved == json.loads(json.dumps(cfg)), f"{exp_id}: 중간 저장과 설정이 다름 → {out} 폴더를 지우거나 설정을 되돌리세요"
    json.dump(cfg, open(out / "cfg.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    seed_everything(cfg["seed"])

    sets = {"train": load_train_df(cfg), "val": val_df, "val_ext": val_ext, "val_hard": val_hard}
    resize = cfg.get("resize", "squash")      # 'squash'(찌그러뜨려 맞춤) / 'letterbox'(비율 유지) — 한 실험의 모든 세트에 같은 방식
    sets = {k: cache_resized(_subset(df, cfg["seed"]), CACHE_ROOT, name=k, mode=resize).reset_index(drop=True) for k, df in sets.items()}
    train_tf, eval_tf = build_transforms(cfg["img_size"], cfg["aug"])
    train_loader = make_loader(sets["train"], train_tf, cfg["batch"], True, bg_swap="bg_swap" in cfg["aug"])
    eval_loaders = {k: make_loader(sets[k], eval_tf, cfg["batch"] * 2, False) for k in ["val", "val_ext", "val_hard"]}

    model = build_model(cfg).to(DEVICE)
    criterion, eval_criterion = make_criteria(cfg, sets["train"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs"]) if cfg["scheduler"] == "cosine" else None
    scaler = torch.amp.GradScaler() if DEVICE.type == "cuda" else None

    hist, best, best_ep, bad, start = [], -1.0, 0, 0, 1
    if ckpt_path.exists():   # 끊긴 학습 이어하기
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"])
        optimizer.load_state_dict(ck["optimizer"])
        if scheduler:
            scheduler.load_state_dict(ck["scheduler"])
        if scaler and ck["scaler"]:
            scaler.load_state_dict(ck["scaler"])
        hist, best, best_ep, bad, start = ck["hist"], ck["best"], ck["best_ep"], ck["bad"], ck["epoch"] + 1
        seed_everything(cfg["seed"] + 1000 * start)   # 이어 한 epoch가 첫 epoch와 같은 순서를 반복하지 않게
        print(f"[{exp_id}] epoch {start}부터 이어서 학습")
    for ep in range(start, cfg["epochs"] + 1):
        t0 = time.perf_counter()
        lr = optimizer.param_groups[0]["lr"]            # 이 epoch에 쓰는 학습률
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, cfg.get("batch_mix", False))
        va_loss, va_acc, _, _ = evaluate(model, eval_loaders["val"], eval_criterion)
        ex_loss, ex_acc, _, _ = evaluate(model, eval_loaders["val_ext"], eval_criterion)
        if scheduler:
            scheduler.step()
        score = round((va_acc + ex_acc) / 2, 6)   # 소수점 오차로 같은 점수가 갈리지 않게 반올림 (동점이면 앞 epoch 유지)
        hist.append(dict(epoch=ep, train_loss=tr_loss, train_acc=tr_acc, val_loss=va_loss, val_acc=va_acc,
                         val_ext_loss=ex_loss, val_ext_acc=ex_acc, score=score, lr=lr, sec=time.perf_counter() - t0))
        pd.DataFrame(hist).to_csv(out / "history.csv", index=False)
        print(f"[{exp_id}] ep{ep:03d} train {tr_loss:.3f}/{tr_acc:.3f} val {va_loss:.3f}/{va_acc:.3f} "
              f"val_ext {ex_acc:.3f} 기준 {score:.4f} lr {lr:.2e} ({hist[-1]['sec']:.0f}s)")
        if score > best or cfg.get("select") == "last":   # select="last": 항상 마지막 epoch 가중치 사용 (이어 학습용)
            best, best_ep, bad = score, ep, 0
            torch.save(model.state_dict(), out / "best.pth")
        else:
            bad += 1
            if bad >= cfg["patience"]:
                print(f"[{exp_id}] 조기 종료 (patience {cfg['patience']})")
                break
        tmp = ckpt_path.with_suffix(".tmp")
        torch.save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(), epoch=ep, hist=hist, best=best, best_ep=best_ep, bad=bad,
                        scheduler=scheduler.state_dict() if scheduler else None, scaler=scaler.state_dict() if scaler else None), tmp)
        tmp.replace(ckpt_path)

    model.load_state_dict(torch.load(out / "best.pth", map_location=DEVICE))
    acc, per_class, by_corr = {}, {}, {}
    by_source, metal_bg, per_class_ext = {}, None, {}
    for k, loader in eval_loaders.items():
        _, acc[k], p, y = evaluate(model, loader, eval_criterion)
        if k == "val":
            per_class = {c: float((p[y == i] == i).mean()) if (y == i).any() else float("nan") for i, c in enumerate(CLASSES)}
        if k == "val_ext":
            ve = sets["val_ext"]
            ok = pd.Series(p == y)
            by_source = {g: float(ok[idx].mean()) for g, idx in ve.groupby("source").groups.items()}
            per_class_ext = {c: float((p[y == i] == i).mean()) for i, c in enumerate(CLASSES) if (y == i).any()}
            metal = ((ve.source == "aihub") & (ve.label == "metal")).to_numpy()
            if metal.any():   # 배경 점검: AI Hub 금속 사진을 battery·유리로 읽은 비율 (없으면 None)
                bad_ids = [CLASSES.index(c) for c in ["battery", "brown-glass", "green-glass", "white-glass"]]
                metal_bg = float(pd.Series(p[metal]).isin(bad_ids).mean())
        if k == "val_hard":
            ok = pd.Series(p == y)
            by_corr = {g: float(ok[idx].mean()) for g, idx in sets["val_hard"].groupby("corruption").groups.items()}
    h = pd.DataFrame(hist)
    metrics = dict(val=acc["val"], val_ext=acc["val_ext"], val_hard=acc["val_hard"],
                   val_avg=(acc["val"] + acc["val_ext"] + acc["val_hard"]) / 3,
                   per_class_val=per_class, by_corruption=by_corr,
                   val_ext_by_source=by_source, aihub_metal_as_glass_battery=metal_bg, per_class_val_ext=per_class_ext,
                   best_epoch=best_ep, epochs_run=len(h),
                   minutes=round(float(h["sec"].sum()) / 60, 2))
    metrics = json.loads(json.dumps(metrics))              # 저장 후 다시 읽은 값과 같은 형태로
    json.dump(metrics, open(out / "metrics.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    ckpt_path.unlink(missing_ok=True)                      # 결과를 저장했으니 중간 저장은 지움
    if "val_dirty" in globals():                           # 손상 검증셋이 있으면 3차 지표도 (실패해도 학습 결과는 이미 저장됨)
        try:
            metrics.update(json.loads(json.dumps(extra_metrics(model, cfg, metrics))))
            json.dump(metrics, open(out / "metrics.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
        except Exception as e:
            print(f"[{exp_id}] 3차 지표 계산 실패 — 다음 실행 때 rescore_experiment가 학습 없이 채움: {e}")
    print(f"[{exp_id}] 완료: val {metrics['val']:.4f} / val_ext {metrics['val_ext']:.4f} / val_hard {metrics['val_hard']:.4f} "
          f"→ val_avg {metrics['val_avg']:.4f} (최고 epoch {metrics['best_epoch']}, {metrics['minutes']}분)")
    return metrics


EXP_ROOT = experiment_root(IN_COLAB)
print("실험 폴더:", EXP_ROOT, "| 장치:", DEVICE)

In [ ]:
# [셀 목적] 다른 계정에 있는 이전 실험 결과(F3 등)를 공개 링크로 받아 오는 함수 (PREV_RUNS_FOLDER_ID를 쓸 때만)
def pull_prev_runs(folder_id: str, dst_root: Path):
    """이전 계정의 공개 결과 폴더(garbage_classification_runs)에서 experiments/·final_test/를 받아 결과 폴더로 복사 (덮어쓰지 않음)"""
    tmp = Path("/content/prev_runs")
    download_public_folder(folder_id, tmp)
    seed_results(tmp, dst_root)

## 6. S1 학습

In [ ]:
# [셀 목적] 자르기 평가 함수 정의, S1 설정, RUN_S1·RUN_S0 이면 학습하고 원본·70%·50% 자르기 성능을 출력
def assert_full_data(sets: dict):
    """실험 사다리는 모든 이미지가 있을 때만 실행 (빠지면 실험마다 데이터가 달라져 비교가 무의미)"""
    missing = {k: int((~df["path"].map(lambda p: (ROOT / p).exists())).sum()) for k, df in sets.items()}
    missing = {k: v for k, v in missing.items() if v}
    if missing:
        raise RuntimeError(f"이미지가 없음 {missing} — AI Hub 등 전체 데이터가 있는 환경(내 Drive 연결)에서 실행하세요")


class CropEval(Dataset):
    """잘린 사진 평가용: 사진마다 정해진 무작위 위치(경로로 정함)에서 가로·세로 frac 비율로 잘라 transform 적용"""
    def __init__(self, df: pd.DataFrame, transform, frac: float):
        self.files, self.paths = df["file"].tolist(), df["path"].tolist()
        self.labels = [CLASS_TO_IDX[c] for c in df["label"]]
        self.transform, self.frac = transform, frac

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert("RGB")
        r = random.Random(zlib.crc32(self.paths[i].encode("utf-8")))
        w, h = img.size
        cw, ch = int(w * self.frac), int(h * self.frac)
        x, y = r.randint(0, w - cw), r.randint(0, h - ch)
        return self.transform(img.crop((x, y, x + cw, y + ch))), self.labels[i]


def crop_report(exp_id: str, exp_root: Path) -> dict:
    """저장된 best.pth로 원본·70%·50% 자르기 조건의 정확도와 손실(val·val_ext 평균)을 잰다"""
    cfg = json.load(open(Path(exp_root) / exp_id / "cfg.json", encoding="utf-8"))
    model = build_model(cfg).to(DEVICE)
    model.load_state_dict(torch.load(Path(exp_root) / exp_id / "best.pth", map_location=DEVICE))
    _, eval_tf = build_transforms(cfg["img_size"], [])
    row = {"id": exp_id}
    for name, frac in [("원본", 1.0), ("crop70", 0.7), ("val_crop", 0.5)]:
        res = []
        for k, df in [("val", val_df), ("val_ext", val_ext)]:
            d = cache_resized(_subset(df, cfg["seed"]), CACHE_ROOT, name=k).reset_index(drop=True)
            loader = DataLoader(CropEval(d, eval_tf, frac), batch_size=cfg["batch"] * 2, shuffle=False,
                                num_workers=2 if IN_COLAB else 0)
            res.append(evaluate(model, loader, nn.CrossEntropyLoss())[:2])
        row[name + "_손실"], row[name] = np.mean(res, axis=0)
    return row


S1_CFG = dict(width=48, hidden=[], depths=[2, 2, 3, 3], pool="avgmax", img_size=128,
              aug=["dark_blur", "hflip", "vflip", "rot", "rrc_wide"], batch_mix=True,
              train_data="mix2", max_per_class=1000, epochs=S_EPOCHS, patience=S_EPOCHS, batch=64,
              lr=1e-3, weight_decay=1e-4, scheduler="cosine", label_smoothing=0.1, seed=7, select="last",
              weak_weight=WEAK_WEIGHT, weak_classes=WEAK_CLASSES)
SUFFIX = "" if S_EPOCHS == 60 else f"_{S_EPOCHS}ep"
S1_ID, S0_ID = "S1" + SUFFIX, "S0" + SUFFIX
for exp_id, go, cfg in [(S1_ID, RUN_S1, S1_CFG), (S0_ID, RUN_S0, {**S1_CFG, "weak_weight": 1.0})]:
    if go:
        assert_full_data({"학습 데이터": train_mix, "val": val_df, "val_ext": val_ext, "val_hard": val_hard})
        run_experiment(exp_id, cfg, EXP_ROOT)
        rep = pd.DataFrame([crop_report(exp_id, EXP_ROOT)]).set_index("id")
        rep.to_csv(EXP_ROOT / exp_id / "crop_report.csv", encoding="utf-8-sig")
        display(rep.round(4))

In [ ]:
# [셀 목적] S1 학습 기록 그래프 (손실·정확도)
if (EXP_ROOT / S1_ID / "history.csv").exists():
    h = pd.read_csv(EXP_ROOT / S1_ID / "history.csv")
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for k, n in [("train", "학습"), ("val", "val"), ("val_ext", "val_ext")]:
        ax[0].plot(h.epoch, h[f"{k}_loss"], label=n)
        ax[1].plot(h.epoch, h[f"{k}_acc"], label=n)
    ax[0].set(title="손실 (학습은 가중치·스무딩·섞기 포함이라 val과 직접 비교 불가)", xlabel="epoch")
    ax[1].set(title="정확도", xlabel="epoch")
    for a in ax:
        a.legend()
        a.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 7. F3 + S1 앙상블
비율은 약한 4클래스용·나머지 8클래스용 두 개만 쓴다. val·val_ext를 절반(A·B)으로 나눠 A에서 고르고 B에서 확인한다.
채택(B, F3+TTA 대비): 50% 자르기 +0.01 이상, 원본 하락 0.01 이내, 나머지 8클래스 평균 재현율 하락 0.01 이내.

In [ ]:
# [셀 목적] F3과 S1의 확률을 클래스 묶음별 비율로 합침 — 검증 사진 절반에서 비율을 고르고 나머지 절반에서 F3+TTA와 비교, 채택 여부 출력
@torch.no_grad()
def tta_probs(exp_id: str, frac: float):
    """저장된 모델의 좌우 반전 TTA 확률. 반환: 확률, 정답 번호, 세트 번호(0 val, 1 val_ext), 경로"""
    cfg = json.load(open(EXP_ROOT / exp_id / "cfg.json", encoding="utf-8"))
    model = build_model(cfg).to(DEVICE)
    model.load_state_dict(torch.load(EXP_ROOT / exp_id / "best.pth", map_location=DEVICE))
    model.eval()
    _, eval_tf = build_transforms(cfg["img_size"], [])
    P, Y, S, paths = [], [], [], []
    for s, (k, df) in enumerate([("val", val_df), ("val_ext", val_ext)]):
        d = cache_resized(_subset(df, 42), CACHE_ROOT, name=k).reset_index(drop=True)
        for x, y in DataLoader(CropEval(d, eval_tf, frac), batch_size=128, shuffle=False, num_workers=2 if IN_COLAB else 0):
            x = x.to(DEVICE)
            P.append(((model(x) + model(torch.flip(x, [3]))) / 2).softmax(1).double().cpu().numpy())
            Y.append(y.numpy())
        S += [s] * len(d)
        paths += d.path.tolist()
    return np.concatenate(P), np.concatenate(Y), np.array(S), paths


def mix_probs(p3, p1, w_weak: float, w_strong: float):
    """점수_c = (1-w_c)·F3 + w_c·S1 → 합이 1이 되게 나눔 (w는 약한 클래스용·나머지용 두 개)"""
    w = np.array([w_weak if c in WEAK_CLASSES else w_strong for c in CLASSES])
    q = (1 - w) * p3 + w * p1
    return q / q.sum(1, keepdims=True).clip(1e-12)


def score(q, y, s, m):
    """세트(val·val_ext)별 정확도와 손실의 평균 (m: 쓸 사진)"""
    acc, loss = [], []
    for k in (0, 1):
        t = m & (s == k)
        acc.append((q[t].argmax(1) == y[t]).mean())
        loss.append(-np.log(q[t, y[t]].clip(1e-12)).mean())
    return float(np.mean(acc)), float(np.mean(loss))


def recall(q, y, m):
    """클래스별 재현율 (val·val_ext 합)"""
    return np.array([(q[m & (y == k)].argmax(1) == k).mean() if (m & (y == k)).any() else np.nan for k in range(len(CLASSES))])


if RUN_ENSEMBLE:
    if PREV_RUNS_FOLDER_ID and not (EXP_ROOT / "F3" / "best.pth").exists():
        pull_prev_runs(PREV_RUNS_FOLDER_ID, EXP_ROOT.parent)
    for e in ("F3", S1_ID):
        assert (EXP_ROOT / e / "metrics.json").exists(), f"{e} 결과가 없음: {EXP_ROOT / e} — 학습을 먼저 끝내거나 PREV_RUNS_FOLDER_ID를 확인하세요"
    R = {}
    for name, frac in [("원본", 1.0), ("crop50", 0.5)]:
        p3, y, s, paths = tta_probs("F3", frac)
        p1, y1, s1, paths1 = tta_probs(S1_ID, frac)
        assert paths == paths1 and (y == y1).all()
        R[name] = (p3, p1)
    half = np.array([zlib.crc32(("half:" + p).encode("utf-8")) % 2 for p in paths])
    A, B = half == 0, half == 1                    # A: 비율 고르기, B: 확인
    grid = [round(0.1 * i, 1) for i in range(11)]
    best = None
    for w_weak in grid:
        for w_strong in grid:
            sc = [score(mix_probs(*R[n], w_weak, w_strong), y, s, A) for n in R]
            key = (round(float(np.mean([a for a, _ in sc])), 6), -float(np.mean([l for _, l in sc])))   # 정확도 → 동률이면 손실이 작은 쪽
            if best is None or key > best[0]:
                best = (key, w_weak, w_strong)
    _, W_WEAK, W_STRONG = best
    print(f"A에서 고른 S1 비율: 약한 클래스 {W_WEAK}, 나머지 {W_STRONG}  (A {A.sum()}장, B {B.sum()}장)")
    rows, rec = [], {}
    for label, w in [("F3+TTA", (0.0, 0.0)), (S1_ID + "+TTA", (1.0, 1.0)), ("앙상블", (W_WEAK, W_STRONG)), ("단순 평균(0.5, 0.5)", (0.5, 0.5))]:
        row = {"모델": label}
        for n in R:
            q = mix_probs(*R[n], *w)
            row[n], row[n + "_손실"] = score(q, y, s, B)
            if n == "crop50":
                rec[label] = recall(q, y, B)
        rows.append(row)
    table = pd.DataFrame(rows).set_index("모델")
    rt = pd.DataFrame(rec, index=CLASSES)
    rt.insert(0, "약한 클래스", [c in WEAK_CLASSES for c in CLASSES])
    strong = ~rt["약한 클래스"]
    d_crop = table.loc["앙상블", "crop50"] - table.loc["F3+TTA", "crop50"]
    d_orig = table.loc["앙상블", "원본"] - table.loc["F3+TTA", "원본"]
    d_strong = rt.loc[strong, "앙상블"].mean() - rt.loc[strong, "F3+TTA"].mean()
    d_weak = rt.loc[~strong, "앙상블"].mean() - rt.loc[~strong, "F3+TTA"].mean()
    ok = bool(d_crop >= 0.01 and d_orig >= -0.01 and d_strong >= -0.01)
    print("B(고르기에 쓰지 않은 절반)의 결과")
    display(table.round(4))
    print("B, 50% 자르기 클래스별 재현율")
    display(rt.round(3))
    print(f"50% 자르기 {d_crop:+.4f} / 원본 {d_orig:+.4f} / 나머지 8클래스 재현율 {d_strong:+.4f} / 약한 4클래스 재현율 {d_weak:+.4f} →",
          "채택" if ok else "채택 안 함")
    ENS_DIR = EXP_ROOT / (S1_ID + "_ensemble")
    ENS_DIR.mkdir(parents=True, exist_ok=True)
    table.to_csv(ENS_DIR / "summary.csv", encoding="utf-8-sig")
    rt.to_csv(ENS_DIR / "recall_crop50.csv", encoding="utf-8-sig")
    json.dump(dict(w_weak=W_WEAK, w_strong=W_STRONG, weak_classes=WEAK_CLASSES, d_crop=float(d_crop), d_orig=float(d_orig),
                   d_strong=float(d_strong), d_weak=float(d_weak), adopted=ok),
              open(ENS_DIR / "result.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print("저장:", ENS_DIR)